[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megusto0/mp-2/blob/main/notebooks/02_simplex_max.ipynb)

# Симплекс-метод для максимума Z

Ноутбук показывает двухфазный симплекс-метод без клонирования репозитория.

In [ ]:
import numpy as np
import pandas as pd

raw_constraints = [
    (np.array([5.0, -2.0]), "<=", 4.0),
    (np.array([1.0, -2.0]), ">=", -4.0),
    (np.array([1.0, 1.0]), ">=", 4.0),
]

def normalize(a, sense, b):
    if b >= 0:
        return a.astype(float), sense, float(b)
    return -a.astype(float), {"<=": ">=", ">=": "<=", "=": "="}[sense], float(-b)

def standardize(raw):
    names = ["x1", "x2"]
    rows, rhs, basis, artificial = [], [], [], []
    for idx, item in enumerate(raw, start=1):
        a, sense, b = normalize(*item)
        row = [0.0] * len(names)
        row[0], row[1] = a
        if sense == "<=":
            names.append(f"s{idx}")
            row.append(1.0)
            for old in rows:
                old.append(0.0)
            basis.append(len(names) - 1)
        elif sense == ">=":
            names.append(f"e{idx}")
            row.append(-1.0)
            for old in rows:
                old.append(0.0)
            names.append(f"a{idx}")
            row.append(1.0)
            for old in rows:
                old.append(0.0)
            basis.append(len(names) - 1)
            artificial.append(len(names) - 1)
        rows.append(row)
        rhs.append(b)
    return names, np.array(rows), np.array(rhs), basis, artificial

def canonical(A, b, basis, objective):
    B = A[:, basis]
    inv = np.linalg.inv(B)
    table = inv @ A
    right = inv @ b
    reduced = objective - objective[basis] @ table
    value = float(objective[basis] @ right)
    return table, right, reduced, value

def simplex(A, b, basis, objective, names, phase):
    steps = []
    basis = list(basis)
    for iteration in range(30):
        table, right, reduced, value = canonical(A, b, basis, objective)
        nonbasis = [idx for idx in range(len(names)) if idx not in basis]
        candidates = [idx for idx in nonbasis if reduced[idx] > 1e-9]
        entering = max(candidates, key=lambda idx: (reduced[idx], -idx)) if candidates else None
        leaving_row = None
        if entering is not None:
            ratios = [(right[row] / table[row, entering], row) for row in range(A.shape[0]) if table[row, entering] > 1e-9]
            leaving_row = min(ratios)[1]
        steps.append({
            "phase": phase,
            "iteration": iteration,
            "basis": ", ".join(names[idx] for idx in basis),
            "rhs": "; ".join(f"{v:.6g}" for v in right),
            "objective": value,
            "entering": names[entering] if entering is not None else "",
            "leaving": names[basis[leaving_row]] if leaving_row is not None else "",
        })
        if entering is None:
            return pd.DataFrame(steps), basis, table, right, reduced, value
        basis[leaving_row] = entering
    raise RuntimeError("Simplex did not converge")

names, A, b, basis, artificial = standardize(raw_constraints)
phase1_objective = np.zeros(len(names))
for idx in artificial:
    phase1_objective[idx] = -1.0

phase1, phase1_basis, *_ = simplex(A, b, basis, phase1_objective, names, "phase I")
keep = [idx for idx in range(len(names)) if idx not in artificial]
new_index = {old: new for new, old in enumerate(keep)}
names2 = [names[idx] for idx in keep]
A2 = A[:, keep]
basis2 = [new_index[idx] for idx in phase1_basis]

## Фаза I

Искусственная переменная нужна только для ограничения `x1 + x2 >= 4`. Фаза I ищет начальную допустимую базисную точку.

In [ ]:
phase1

## Фаза II

Теперь целевая функция заменяется на нужную для текущей задачи.

In [ ]:
objective2 = np.array([-3.0, 6.0, 0.0, 0.0, 0.0])
phase2, final_basis, table, right, reduced, value = simplex(A2, b, basis2, objective2, names2, 'phase II')
phase2

In [ ]:
solution = dict.fromkeys(names2, 0.0)
for name, val in zip([names2[idx] for idx in final_basis], right):
    solution[name] = val
x = np.array([solution['x1'], solution['x2']])
Z = -3*x[0] + 6*x[1]
print('x* =', x)
print('Z =', Z)
print('transformed objective =', value)
print('reduced costs =', dict(zip(names2, reduced)))

В последней строке видно, что для максимума есть нулевая оценка у небазисной переменной. Это означает не единственную точку максимума.